<a href="https://colab.research.google.com/github/Codemioo/bioinformatic/blob/main/mini_blast_search_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def create_kmer_index(db_sequence: str, k: int = 3) -> dict:
    """
    Veritabanı dizisini k-mer'lere böler ve her k-mer'in
    başlangıç indekslerini bir sözlükte (Hash Table) saklar.
    """
    index = {}
    for i in range(len(db_sequence) - k + 1):
        kmer = db_sequence[i:i+k].upper()
        if kmer not in index:
            index[kmer] = []
        index[kmer].append(i)
    return index

def find_seeds(query_sequence: str, kmer_index: dict, k: int = 3) -> list:
    """
    Sorgu dizisini k-mer'lere böler ve veritabanı indeksinde
    eşleşen tohumları (seed/hit) tespit eder.
    """
    seeds = []
    for i in range(len(query_sequence) - k + 1):
        kmer = query_sequence[i:i+k].upper()
        if kmer in kmer_index:
            for db_pos in kmer_index[kmer]:
                # (Query Başlangıç Pozisyonu, DB Başlangıç Pozisyonu, k-mer dizisi)
                seeds.append((i, db_pos, kmer))
    return seeds

# Örnek Veritabanı ve Sorgu Dizisi
db_seq = "ATGCGATCGATCGATCGATCGAATTCGATCG"
query_seq = "GAATTCG"

# 1. Veritabanını 3-mer (k=3) olarak indeksleme
k_size = 3
db_index = create_kmer_index(db_seq, k=k_size)

# 2. Tohumları (Seeds) Bulma
found_seeds = find_seeds(query_seq, db_index, k=k_size)

print(f"Veritabanı İndeksi (İlk 5 k-mer): {dict(list(db_index.items())[:5])}\n")
print(f"Bulunan Tohum Sayısı (Seeds): {len(found_seeds)}")
print("Eşleşen İlk Tohumlar (Query Pos, DB Pos, K-mer):")
for seed in found_seeds[:5]:
    print(seed)

Veritabanı İndeksi (İlk 5 k-mer): {'ATG': [0], 'TGC': [1], 'GCG': [2], 'CGA': [3, 7, 11, 15, 19, 25], 'GAT': [4, 8, 12, 16, 26]}

Bulunan Tohum Sayısı (Seeds): 10
Eşleşen İlk Tohumlar (Query Pos, DB Pos, K-mer):
(0, 20, 'GAA')
(1, 21, 'AAT')
(2, 22, 'ATT')
(3, 23, 'TTC')
(4, 6, 'TCG')


In [2]:
def extend_seed(query_seq: str, db_seq: str, seed: tuple, match_score: int = 1, mismatch_score: int = -1, dropoff: int = 2):
    """
    Bulunan bir tohumu sağa ve sola doğru genişleterek High-scoring Segment Pair (HSP) oluşturur.
    """
    q_start, db_start, kmer = seed
    k = len(kmer)

    # Tohumun ilk skoru
    current_score = k * match_score
    max_score = current_score

    # 1. SOLA DOĞRU UZATMA (Left Extension)
    q_left = q_start - 1
    db_left = db_start - 1
    left_q_ext, left_db_ext = "", ""

    while q_left >= 0 and db_left >= 0:
        if query_seq[q_left] == db_seq[db_left]:
            current_score += match_score
        else:
            current_score += mismatch_score

        if current_score > max_score:
            max_score = current_score
        elif max_score - current_score >= dropoff:
            break  # Skor çok düştü, uzatmayı kes

        left_q_ext = query_seq[q_left] + left_q_ext
        left_db_ext = db_seq[db_left] + left_db_ext
        q_left -= 1
        db_left -= 1

    # Skor sıfırlama (Sağa uzatma için güncel skoru sıfırdan hesapla)
    current_score = max_score

    # 2. SAĞA DOĞRU UZATMA (Right Extension)
    q_right = q_start + k
    db_right = db_start + k
    right_q_ext, right_db_ext = "", ""

    while q_right < len(query_seq) and db_right < len(db_seq):
        if query_seq[q_right] == db_seq[db_right]:
            current_score += match_score
        else:
            current_score += mismatch_score

        if current_score > max_score:
            max_score = current_score
        elif max_score - current_score >= dropoff:
            break  # Skor çok düştü, uzatmayı kes

        right_q_ext += query_seq[q_right]
        right_db_ext += db_seq[db_right]
        q_right += 1
        db_right += 1

    # Tam Hizalanmış Parçalar (HSP)
    aligned_query = left_q_ext + kmer + right_q_ext
    aligned_db = left_db_ext + kmer + right_db_ext

    return aligned_query, aligned_db, max_score


# İlk bulunan tohumu genişletme
if found_seeds:
    first_seed = found_seeds[0]
    aln_q, aln_db, score = extend_seed(query_seq, db_seq, first_seed)

    print("--- TOHUM GENİŞLETME SONUCU (HSP) ---")
    print(f"Tohum (Seed) : {first_seed[2]} (Query Pos: {first_seed[0]}, DB Pos: {first_seed[1]})")
    print(f"Sorgu (Query): {aln_q}")
    print(f"Veritabanı   : {aln_db}")
    print(f"En Yüksek Skor: {score}")

--- TOHUM GENİŞLETME SONUCU (HSP) ---
Tohum (Seed) : GAA (Query Pos: 0, DB Pos: 20)
Sorgu (Query): GAATTCG
Veritabanı   : GAATTCG
En Yüksek Skor: 7


In [3]:
import math

def calculate_stats(raw_score: int, query_len: int, db_len: int, K: float = 0.13, lambda_param: float = 0.318):
    """Karlin-Altschul istatistiği ile Bit Score ve E-Value hesaplar."""
    # Bit Score hesaplama
    bit_score = (lambda_param * raw_score - math.log(K)) / math.log(2)

    # E-Value hesaplama
    e_value = query_len * db_len * (2 ** (-bit_score))
    return round(bit_score, 2), e_value

def display_blast_alignment(query_aln: str, db_aln: str, raw_score: int, q_len: int, db_len: int):
    """BLAST tarzı görsel hizalama ve istatistik raporu sunar."""
    match_line = ""
    identity_count = 0

    for q_char, db_char in zip(query_aln, db_aln):
        if q_char == db_char:
            match_line += "|"
            identity_count += 1
        else:
            match_line += "."

    identity_pct = (identity_count / len(query_aln)) * 100
    bit_score, e_value = calculate_stats(raw_score, q_len, db_len)

    print("=" * 55)
    print("               BLAST-LIKE SEARCH REPORT             ")
    print("=" * 55)
    print(f"HSP Raw Score : {raw_score}")
    print(f"Bit Score     : {bit_score} bits")
    print(f"E-Value       : {e_value:.2e}")
    print(f"Identity      : {identity_count}/{len(query_aln)} (%{identity_pct:.1f})")
    print("-" * 55)
    print(f"Query : {query_aln}")
    print(f"        {match_line}")
    print(f"Sbjct : {db_aln}")
    print("=" * 55)



# 3. Adımdan elde ettiğimiz aln_q, aln_db ve score değerlerini kullanalım
display_blast_alignment(
    query_aln=aln_q,
    db_aln=aln_db,
    raw_score=score,
    q_len=len(query_seq),
    db_len=len(db_seq)
)

               BLAST-LIKE SEARCH REPORT             
HSP Raw Score : 7
Bit Score     : 6.15 bits
E-Value       : 3.05e+00
Identity      : 7/7 (%100.0)
-------------------------------------------------------
Query : GAATTCG
        |||||||
Sbjct : GAATTCG
